In [ ]:

import os
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from IPython.display import display, Markdown
from ultralytics import YOLO
import cv2


plt.rcParams["figure.figsize"] = (10, 8)

# Fase di Preprocessing:

Fase di estrazione dei frame dai video, e generazione dei txt compatibili con il modello YOLO.

In [ ]:
def convert_xml_and_extract_frames(video_path, xml_file, frames_dir, labels_dir, img_width, img_height, video_name):
    # Parse XML e identifica i frame annotati
    tree = ET.parse(xml_file)
    root = tree.getroot()
    os.makedirs(frames_dir, exist_ok=True)
    os.makedirs(labels_dir, exist_ok=True)

    annotated_frames = {}  # Mappa frame -> annotazioni
    for points in root.findall(".//points"):
        frame = int(points.get("frame"))
        outside = int(points.get("outside"))
        used_in_game = points.find(".//attribute[@name='used_in_game']").text if points.find(".//attribute[@name='used_in_game']") is not None else "1"

        # Ottieni le informazioni di annotazione per ogni frame
        x, y = map(float, points.get("points").split(","))

        # Se "used_in_game" è uguale a 0, il frame deve essere ignorato, quindi non ha annotazioni
        if used_in_game == "0":
            annotated_frames[frame] = ""  # File vuoto
        else:
            # Normalizzazione delle coordinate
            x_center = x / img_width
            y_center = y / img_height
            width = 0.05  # Approssimazione della dimensione
            height = 0.05
            # Se il frame è visibile (outside=0), crea le annotazioni
            if outside == 0:
                annotated_frames[frame] = f"0 {x_center} {y_center} {width} {height}\n"
            else:
                # Se fuori dal campo (outside=1), puoi scrivere un file txt vuoto o con una convenzione YOLO
                annotated_frames[frame] = ""  # File vuoto o con un formato che YOLO comprende come "assenza di oggetti"

    # Estrai i frame dal video e genera i file .txt
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Crea il nome del frame
        frame_name = f"{video_name}_frame{frame_count:06d}.jpg"  # Salva come .jpg
        frame_file_path = os.path.join(frames_dir, frame_name)

        # Salva il frame come immagine JPEG con qualità 90
        cv2.imwrite(frame_file_path, frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])

        # Crea il file .txt (anche per i frame senza annotazioni)
        txt_file_path = os.path.join(labels_dir, f"{video_name}_frame{frame_count:06d}.txt")
        with open(txt_file_path, "w") as f:
            # Se ci sono annotazioni, scrivile nel file, altrimenti crea un file vuoto
            annotation = annotated_frames.get(frame_count, "")
            if annotation == "":  # Se non ci sono annotazioni o se used_in_game=0
                # In YOLO, un file vuoto significa "assenza di oggetti"
                f.write("")  # Scrive un file vuoto o puoi scrivere una convenzione YOLO (ad esempio '0 0 0 0 0')
            else:
                f.write(annotation)  # Scrive la label (con annotazioni)

        frame_count += 1

    cap.release()



Per le operazioni di preparazione del dataset, è stato utilizzato il tool online **Roboflow**, che ha svolto un ruolo fondamentale nel migliorare la qualità e la quantità dei dati.

In particolare è stato effettuato per fare **data augmentation**, tramite il cui è stato possibile ottenere una **triplicazione degli esempi positivi**, andando a generare varianti delle immagini originali con trasformazioni come rotazioni, ridimensionamenti e modifiche di luminosità. 

Queste operazioni hanno consentito di migliorare significativamente la capacità del modello di generalizzare a situazioni diverse.

Le principali effettuate sono le seguenti:
* 50% probability of horizontal flip
* 50% probability of vertical flip
* Equal probability of one of the following 90-degree rotations: none, clockwise, counter-clockwise, upside-down
* Random brigthness adjustment of between -15 and +15 percent
* Random Gaussian blur of between 0 and 2.5 pixels

In [ ]:
def convert_xml_and_extract_frames(video_path, xml_file, frames_dir, labels_dir, img_width, img_height, video_name):
    # Parse XML e identifica i frame annotati
    tree = ET.parse(xml_file)
    root = tree.getroot()
    os.makedirs(frames_dir, exist_ok=True)
    os.makedirs(labels_dir, exist_ok=True)

    annotated_frames = {}  # Mappa frame -> annotazioni
    for points in root.findall(".//points"):
        frame = int(points.get("frame"))
        outside = int(points.get("outside"))
        used_in_game = points.find(".//attribute[@name='used_in_game']").text if points.find(".//attribute[@name='used_in_game']") is not None else "1"

        # Ottieni le informazioni di annotazione per ogni frame
        x, y = map(float, points.get("points").split(","))

        # Se "used_in_game" è uguale a 0, il frame deve essere ignorato, quindi non ha annotazioni
        if used_in_game == "0":
            annotated_frames[frame] = ""  # File vuoto
        else:
            # Normalizzazione delle coordinate
            x_center = x / img_width
            y_center = y / img_height
            width = 0.05  # Approssimazione della dimensione
            height = 0.05
            # Se il frame è visibile (outside=0), crea le annotazioni
            if outside == 0:
                annotated_frames[frame] = f"0 {x_center} {y_center} {width} {height}\n"
            else:
                # Se fuori dal campo (outside=1), puoi scrivere un file txt vuoto o con una convenzione YOLO
                annotated_frames[frame] = ""  # File vuoto o con un formato che YOLO comprende come "assenza di oggetti"

    # Estrai i frame dal video e genera i file .txt
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Crea il nome del frame
        frame_name = f"{video_name}_frame{frame_count:06d}.jpg"  # Salva come .jpg
        frame_file_path = os.path.join(frames_dir, frame_name)

        # Salva il frame come immagine JPEG con qualità 90
        cv2.imwrite(frame_file_path, frame, [int(cv2.IMWRITE_JPEG_QUALITY), 90])

        # Crea il file .txt (anche per i frame senza annotazioni)
        txt_file_path = os.path.join(labels_dir, f"{video_name}_frame{frame_count:06d}.txt")
        with open(txt_file_path, "w") as f:
            # Se ci sono annotazioni, scrivile nel file, altrimenti crea un file vuoto
            annotation = annotated_frames.get(frame_count, "")
            if annotation == "":  # Se non ci sono annotazioni o se used_in_game=0
                # In YOLO, un file vuoto significa "assenza di oggetti"
                f.write("")  # Scrive un file vuoto o puoi scrivere una convenzione YOLO (ad esempio '0 0 0 0 0')
            else:
                f.write(annotation)  # Scrive la label (con annotazioni)

        frame_count += 1

    cap.release()



# Modello:

In [ ]:
model=YOLO('yolo11s.pt')
model.train(
    data=r'/kaggle/input/newdataset/data/data.yaml',  # File YAML del dataset
    epochs=200,  # Numero di epoche
    batch=16,    # Dimensione del batch
    imgsz=1024,  # Dimensione dell'immagine
    patience=5, #Early stop dopo 5 epoche
    conf=0.2, #Abbassiamo la confidenza di predizione a 0.2
    warmup_epochs=3,
    lr0=0.0005, # Learning rate iniziale (più basso per una convergenza più stabile)
    lrf=0.01,   # Learning rate finale
    save=True,     # Salva i pesi
    save_period=5, # Salva ogni 5 epoche
    project=r'/kaggle/working//model/yolo_output_prog',  # Salvataggio modello
    device='cuda', # Usa la GPU
    workers=4,         # Numero di workers per il caricamento
    verbose=True
)

In [ ]:
class TrackingEvaluator:
    def __init__(self, gt_ann_file, pred_file):
        self.gt_ann_file = gt_ann_file
        self.pred_file = pred_file
        self.gt_data_points = {}
        self.pred_points = {}
        self.frames_idx = None

    @staticmethod
    def extract_points_data(xml_content):
        root = ET.fromstring(xml_content)
        points_data = {}

        for track in root.findall(".//track"):
            for point in track.findall("points"):
                data = {
                    'frame': int(point.get("frame")),
                    'outside': int(point.get("outside")),
                    'occluded': int(point.get("occluded")),
                    'keyframe': int(point.get("keyframe")),
                    'points': tuple(map(float, point.get("points").split(","))),
                    'z_order': int(point.get("z_order")),
                }
                if data['frame'] in points_data:
                    print(f'Alert: multiple frame entries for ID {data["frame"]}')
                points_data[data['frame']] = data

        return points_data

    @staticmethod
    def _convert_key(k):
        return int(Path(k).stem)

    def load_data(self):
        # Load ground truth data
        gt_content = Path(self.gt_ann_file).read_text()
        self.gt_data_points = self.extract_points_data(gt_content)

        # Load prediction data
        pred_content = Path(self.pred_file).read_text().replace('-Infinity', '-1').replace('Infinity', '-1')
        raw_pred_points = json.loads(pred_content)
        self.pred_points = {
            self._convert_key(k): v for k, v in raw_pred_points.items() if v['x'] >= 0
        }

    def compute_frame_indices(self):
        ordered_list_pred_frame = sorted(self.pred_points.keys())
        ordered_list_gt_frame = sorted(self.gt_data_points.keys())

        print(f'GT   frames: {ordered_list_gt_frame[0]} - {ordered_list_gt_frame[-1]}')
        print(f'PRED frames: {ordered_list_pred_frame[0]} - {ordered_list_pred_frame[-1]}')

        self.frames_idx = (
            min(ordered_list_gt_frame[0], ordered_list_pred_frame[0]),
            max(ordered_list_gt_frame[-1], ordered_list_pred_frame[-1]),
        )
        print(f'Frame Index Range: {self.frames_idx}')

    @staticmethod
    def is_match(x1, y1, x2, y2, threshold=4):
        p1 = np.array((x1, y1))
        p2 = np.array((x2, y2))
        euclid_dist = np.sqrt(np.dot((p1 - p2).T, (p1 - p2)))
        return euclid_dist < threshold

    def evaluate_metrics(self):
        cnt_match = 0
        cnt_no_match = 0
        cnt_no_pred = 0
        cnt_no_frame = 0

        for i in range(self.frames_idx[0], self.frames_idx[1] + 1):
            if i not in self.gt_data_points:
                cnt_no_frame += 1
                continue
            if i not in self.pred_points:
                cnt_no_pred += 1
                continue

            p1 = self.gt_data_points[i]
            p2 = self.pred_points[i]

            if self.is_match(*p1['points'], p2['x'], p2['y']):
                cnt_match += 1
            else:
                cnt_no_match += 1

        total_frames = len(self.gt_data_points)
        print(f'Total frames: {total_frames}')
        print(f'Total predictions: {len(self.pred_points)}')
        print(f'Matches: {cnt_match} ({cnt_match / total_frames:.3f})')
        print(f'No matches: {cnt_no_match} ({cnt_no_match / total_frames:.3f})')
        print(f'No predictions: {cnt_no_pred} ({cnt_no_pred / total_frames:.3f})')
        print(f'No frame data: {cnt_no_frame} ({cnt_no_frame / (self.frames_idx[1] - self.frames_idx[0] + 1):.3f})')

    def compute_tracking_sequence(self):
        norm_width = 1920
        norm_height = 1080

        gt_seq = []
        pred_seq = []

        for i in range(min(self.gt_data_points.keys()), max(self.gt_data_points.keys()) + 1):
            if i in self.gt_data_points:
                if i not in self.pred_points:
                    pred_seq.append((0, 0))
                else:
                    p2 = self.pred_points[i]
                    pred_seq.append((p2['x'] / norm_width, p2['y'] / norm_height))

                x, y = self.gt_data_points[i]['points']
                gt_seq.append((x / norm_width, y / norm_height))

        return gt_seq, pred_seq

    def compute_mse(self):
        gt_seq, pred_seq = self.compute_tracking_sequence()
        mse = mean_squared_error(gt_seq, pred_seq)
        print(f'Mean Squared Error: {mse}')
        return mse

# Creazione delle predizioni

 Andiamo a creare il file delle predizioni sul test set utilizzando come modello il risultante con i pesi migliori.

In [ ]:
def calcola_pred(model_path, dir_test_frame):
    model = YOLO(model_path)
    test_images = [os.path.join(dir_test_frames, f) for f in os.listdir(dir_test_frames) if f.endswith(('.jpg', '.png'))]
    # Dizionario in cui memorizzo le predizioni in formato JSON
    predictions = {}
    for image_path in test_images:
        # Ci prendiamo il nome del file (ad esempio, "ID-6_frame000645.jpg")
        file_name = os.path.basename(image_path)
        # Esegui la predizione sul frame
        results = model.predict(image_path,imgsz=1024)
        # Controllo se ci sono predizioni (bounding boxes)
        if len(results[0].boxes) > 0:  # Se ci sono delle predizioni (risultati)
            boxes = results[0].boxes.xywh.tolist()  # Bounding boxes
            classes = results[0].boxes.cls.tolist()  # Classi delle predizioni
            names = results[0].names  # Nomi delle classi predette (nel nostro caso solo ball)
            confidences = results[0].boxes.conf.tolist()  # Confidenze delle predizioni

            # Itero sui risultati
            for box, cls, conf in zip(boxes, classes, confidences):
                x_center, y_center, width, height = box  # Coordinate del bounding box
                confidence = conf
                detected_class = cls
                name = names[int(cls)]
    
                # Calcola la posizione del centro del bounding box
                x_center_pixel = x_center
                y_center_pixel = y_center
    
                # Aggiungi il file e la predizione al dizionario
                predictions[file_name] = {"x": float(x_center_pixel), "y": float(y_center_pixel)}  # Converte in float per serializzazione
    
        else: predictions[file_name] = {"x": -1, "y": -1}

    # Salva le predizioni in un file JSON
    predictions_file = '/kaggle/working/model/yolo_output_prog/train2/predictions_final.json'
    with open(predictions_file, 'w') as f:
        json.dump(predictions, f, indent=4)
    
    print(f"Le predizioni sono state salvate in: {predictions_file}")

In [ ]:
model_path = "/kaggle/working/model/yolo_output_prog/train/weights/best.pt"
dir_test_frames = 'kaggle/input/newdataset/data/test/images'

calcola_pred(model_path,dir_test_frames)

## Split delle predizioni

Una volta creato il file con le predizioni generali andiamo a dividere le predizioni del video ID-5 dalle predizioni del video ID-6, in modo da poter andare successivamente a valutare le prestazioni del modello.

In [ ]:
def split_pred(pred_file):
    with open(pred_file, 'r') as f:
        predictions = json.load(f)
    
    
    predictions_video_5 = {}
    predictions_video_6 = {}
    
    
    for frame_name, coords in predictions.items():
        if "ID-5" in frame_name:  
            predictions_video_5[frame_name] = coords
        elif "ID-6" in frame_name:  
            predictions_video_6[frame_name] = coords
    
    
    ID_5_pred_file = '/kaggle/working/model/yolo_output_prog/train/predictions_ID-5_final.json'
    with open(ID_5_pred_file, 'w') as f:
        json.dump(predictions_video_5, f, indent=4)

    ID_6_pred_file = '/kaggle/working/model/yolo_output_prog/train2/predictions_ID-6_final.json'
    with open(ID_6_pred_file, 'w') as f:
        json.dump(predictions_video_6, f, indent=4)
    
    print(f"Predizioni video 5 salvate in: {ID_5_pred_file}")
    print(f"Predizioni video 6 salvate in: {ID_6_pred_file}")

In [ ]:
pred_file = '/kaggle/working/model/yolo_output_prog/train/predictions_final.json'
split_pred(pred_file)

#### Rimozione dei prefissi
Andiamo ora a rimuovere il prefisso dalle predizioni ed andare a creare un nuovo file per le predizioni modificate.

In [ ]:
def mod_predictions(input_file, output_file, prefix):
    with open(input_file, 'r') as f:
        predictions = json.load(f)
    
    modified_predictions = {frame_name.replace(prefix, ''): coords for frame_name, coords in predictions.items()}
    
    with open(output_file, 'w') as f:
        json.dump(modified_predictions, f, indent=4)
    
    print(f"Le predizioni modificate sono state salvate in: {output_file}")

In [ ]:
mod_predictions(
    input_file='/kaggle/working/model/yolo_output_prog/train/predictions_ID-5_final.json',
    output_file='/kaggle/working/model/yolo_output_prog/train/modified_predictions_ID-5_final.json',
    prefix='ID-5_frame'
)

mod_predictions(
    input_file='/kaggle/working/model/yolo_output_prog/train/predictions_ID-6_final.json',
    output_file='/kaggle/working/model/yolo_output_prog/train/modified_predictions_ID-6_final.json',
    prefix='ID-6_frame'
)

# Calcolo della MSE (Mean Square Error)

Andiamo a calcolare il mean square error dal nostro addestramento del modello in modo da andare a verificarne le prestazioni.

In [ ]:
# Percorsi ai tuoi file
gt5 = '/kaggle/input/newdataset/data/testgt/ID-5.xml'  
pred5 = '/kaggle/working/model/yolo_output_prog/train/modified_predictions_ID-5_final.json'  # Predizioni in formato JSON
evaluator5 = TrackingEvaluator(gt5, pred5)
evaluator5.load_data()
evaluator5.compute_frame_indices()
evaluator5.evaluate_metrics()
mse5 = evaluator5.compute_mse()

In [ ]:
gt6 = '/kaggle/input/newdataset/data/testgt/ID-6.xml'  
pred6 = '/kaggle/working/model/yolo_output_prog/train/modified_predictions_ID-6_final.json' 

evaluator6 = TrackingEvaluator(gt6, pred6)
evaluator6.load_data()
evaluator6.compute_frame_indices()
evaluator6.evaluate_metrics()
mse6 = evaluator6.compute_mse()

In [ ]:

media= np.mean([mse5,mse6])
print("MSE medio calcolato =",media)

# Valutazione del Mean Squared Error

|          | *MSE*  |
|:--------:|:------:|
| ID-5     | 0.0652 |
| ID-6     | 0.0404 |
| Mse Medio| 0.0528 |

Come si può evincere il Mean Squared Error risultatne è alto, andiamo quindi ad effettuare delle operazioni di post-processing sui nostri dati per provare ad abbassarlo.

# Post Processing

Definiamo due funzioni per il caricamento e il salvataggio di file .json.

In [ ]:
import json
import os
import glob

def load_json(file_path):
    """Carica un file JSON."""
    with open(file_path, 'r') as f:
        return json.load(f)

def save_json(data, file_path):
    """Salva un file JSON."""
    with open(file_path, 'w') as f:
        json.dump(data, f, indent=4)

## Fase Zero: "Ordinamento dei Frame"

In [ ]:
def extract_frame_number(frame_name):
    """
    Estrai il numero del frame dalla stringa del nome del file.
    """
    return int(frame_name.split('_')[-1].replace('.jpg', '').replace('.png', '').replace('frame', ''))

def sort_predictions(input_file, output_file):
    with open(input_file, 'r') as f:
        predictions = json.load(f)
    
    sorted_predictions = dict(sorted(predictions.items(), key=lambda item: extract_frame_number(item[0])))
    
    with open(output_file, 'w') as f:
        json.dump(sorted_predictions, f, indent=4)
    
    print(f"Predizioni ordinate salvate in: {output_file}")

In [ ]:
sort_predictions(
    input_file='/kaggle/working/model/yolo_output_prog/train/modified_predictions_ID-5_final.json',
    output_file='/kaggle/working/model/yolo_output_prog/train/ID-5_ordinato.json'
)
sort_predictions(
    input_file='/kaggle/working/model/yolo_output_prog/train/modified_predictions_ID-6_final.json',
    output_file='/kaggle/working/model/yolo_output_prog/train/ID-6_ordinato.json'
)

### Primo Step: Eliminazione dei Falsi Positivi  

In questa fase ci concentriamo sull'eliminazione delle predizioni errate e isolate, ovvero rilevamenti che il modello ha generato erroneamente. Per farlo, identifichiamo accumuli di rilevamenti in punti statici e isolati, che non seguono le dinamiche naturali del gioco.  

Poiché il pallone è generalmente visibile nelle aree di maggiore interazione tra i giocatori e nei momenti di gioco attivo, le predizioni in zone poco utilizzate del campo (come il centro fisso del frame o gli angoli statici) risultano incoerenti con il contesto.  

Inoltre, il pallone può temporaneamente scomparire dal campo visivo, ad esempio quando viene calciato fuori dal campo o coperto da oggetti come cartelloni pubblicitari.  

Le predizioni isolate possono derivare da:  
- **Rilevamenti errati** causati da elementi visivamente simili al pallone (es. linee bianche, riflessi di luce).  
- **Errori sistematici** dovuti a carenze nel dataset di training, che portano a falsi positivi in determinate aree.  

Questi falsi positivi introducono rumore nei risultati e riducono la precisione complessiva del modello, rendendo necessaria la loro eliminazione per migliorare l'affidabilità delle predizioni.

In [ ]:
def is_static_block(values, start, end, threshold):
    """Verifica se il blocco è statico."""
    for i in range(start, end - 1):
        if values[i]["x"] != -1 and values[i]["y"] != -1 and values[i + 1]["x"] != -1 and values[i + 1]["y"] != -1:
            dx = abs(values[i]["x"] - values[i + 1]["x"])
            dy = abs(values[i]["y"] - values[i + 1]["y"])
            if dx > threshold or dy > threshold:
                return False
    return True

def has_null_neighbors(values, start, end, min_neighbors):
    """Verifica se ci sono almeno `min_neighbors` frame -1, -1 prima e dopo il blocco."""
    before = all(values[max(0, start - i)]["x"] == -1 and values[max(0, start - i)]["y"] == -1 for i in range(1, min_neighbors + 1))
    after = all(values[min(len(values) - 1, end + i)]["x"] == -1 and values[min(len(values) - 1, end + i)]["y"] == -1 for i in range(1, min_neighbors + 1))
    return before and after

def clean_predictions(input_file, output_file, window_size=40, static_threshold=5, min_neighbors=5):
    """Elabora un file JSON per rimuovere outlier statici."""
    predictions = load_json(input_file)
    frames = list(predictions.keys())
    values = list(predictions.values())
    cleaned_predictions = predictions.copy()

    for i in range(len(values) - window_size + 1):
        start, end = i, i + window_size
        valid_frames = [j for j in range(start, end) if values[j]["x"] != -1 and values[j]["y"] != -1]

        if len(valid_frames) > 2:
            first_valid, last_valid = valid_frames[0], valid_frames[-1]
            if is_static_block(values, first_valid, last_valid, static_threshold) and has_null_neighbors(values, first_valid, last_valid, min_neighbors):
                for j in range(start, end):
                    cleaned_predictions[frames[j]] = {"x": -1, "y": -1}

    save_json(cleaned_predictions, output_file)
    print(f"Predizioni depurate salvate in: {output_file}")

In [ ]:

# Specifica i file da elaborare
input_files = [
    "/kaggle/working/model/yolo_output_prog/train/ID-5_ordinato.json",
    "/kaggle/working/model/yolo_output_prog/train/ID-6_ordinato.json"
]

output_directory = "/kaggle/working/model/yolo_output_prog/train/"
os.makedirs(output_directory, exist_ok=True)

# Elaborazione dei due file
for input_file in input_files:
    file_name = os.path.basename(input_file)
    output_file = os.path.join(output_directory, file_name.replace("_ordinato.json", "_SenzaOutlier.json"))
    clean_predictions(input_file, output_file)


### **Secondo Step: Riduzione dei Falsi Negativi**  

Per ridurre i falsi negativi utilizziamo il **Metodo di Interpolazione Lineare**, che aiuta a ricostruire la traiettoria del pallone nei momenti in cui il tracking fallisce.  

🔹 L'interpolazione calcola le coordinate mancanti tra:  
- L’ultimo frame prima della perdita del pallone.  
- Il primo frame dopo la ricomparsa.  
- La traiettoria viene stimata dividendo lo spostamento in intervalli uguali per ogni frame assente.  

 **Limite di 20 frame**: Oltre questo intervallo, l’interpolazione diventa meno affidabile perché il pallone potrebbe aver cambiato direzione o essere stato influenzato da fattori esterni.

In [ ]:
def interpolate_frames(start_coords, end_coords, gap_length):
    """Interpolazione lineare tra due estremi."""
    interpolated = []
    x_step = (end_coords["x"] - start_coords["x"]) / (gap_length + 1)
    y_step = (end_coords["y"] - start_coords["y"]) / (gap_length + 1)
    for i in range(1, gap_length + 1):
        interpolated.append({
            "x": start_coords["x"] + i * x_step,
            "y": start_coords["y"] + i * y_step
        })
    return interpolated

def interpolate_predictions(input_file, output_file, max_gap=20):
    """Interpolazione dei buchi nei dati JSON."""
    predictions = load_json(input_file)
    frames = list(predictions.keys())
    values = list(predictions.values())
    cleaned_predictions = predictions.copy()
    
    i = 0
    while i < len(values):
        if values[i]["x"] != -1 and values[i]["y"] != -1:
            start_index = i
            i += 1
            while i < len(values) and values[i]["x"] == -1 and values[i]["y"] == -1:
                i += 1
            end_index = i

            gap_length = end_index - start_index - 1
            if gap_length > 0 and gap_length <= max_gap and end_index < len(values):
                start_coords = values[start_index]
                end_coords = values[end_index]
                interpolated = interpolate_frames(start_coords, end_coords, gap_length)
                for j in range(gap_length):
                    frame_index = start_index + j + 1
                    cleaned_predictions[frames[frame_index]] = interpolated[j]
        else:
            i += 1

    save_json(cleaned_predictions, output_file)
    print(f"Predizioni interpolate salvate in: {output_file}")

def process_selected_files(input_files, output_directory, **kwargs):
    """Elabora una lista di file specifici."""
    os.makedirs(output_directory, exist_ok=True)
    
    for input_file in input_files:
        file_name = os.path.basename(input_file)
        output_file = os.path.join(output_directory, file_name.replace("_SenzaOutlier.json", "_Interpolato.json"))
        interpolate_predictions(input_file, output_file, **kwargs)

In [ ]:
input_files = [
    "/kaggle/working/model/yolo_output_prog/train/ID-5_SenzaOutlier.json",
    "/kaggle/working/model/yolo_output_prog/train/ID-6_SenzaOutlier.json"
]

output_directory = "/kaggle/working/model/yolo_output_prog/train/"
process_selected_files(input_files, output_directory)

### **Edge Handling**  

Questa tecnica viene utilizzata per gestire predizioni che si interrompono vicino al bordo dell’immagine, ma senza raggiungerlo completamente.  

La applichiamo perché, in molti casi, il pallone sta realmente uscendo dal campo visivo, ma la predizione si ferma troppo presto.  

- **Scenario realistico**: Il pallone spesso si muove verso i bordi durante tiri, passaggi lunghi o azioni rapide.  
- **Soluzione**: Calcoliamo velocità e direzione dai frame precedenti per estendere gradualmente la traiettoria fino al bordo, garantendo un tracciamento più fluido e coerente.

In [ ]:
def is_out_of_bounds(x, y, img_width, img_height):
    """Verifica se una posizione è fuori dai bordi."""
    return x < 0 or x > img_width or y < 0 or y > img_height

def extend_predictions(input_file, output_file, img_width=1920, img_height=1080):
    """Estende i blocchi validi delle predizioni."""
    predictions = load_json(input_file)
    frames = list(predictions.keys())
    values = list(predictions.values())
    cleaned_predictions = predictions.copy()

    i = 0
    while i < len(values):
        if values[i]["x"] != -1 and values[i]["y"] != -1:
            start_index = i
            while i < len(values) and values[i]["x"] != -1 and values[i]["y"] != -1:
                i += 1
            end_index = i - 1

            last_coords = values[end_index]
            if not is_out_of_bounds(last_coords["x"], last_coords["y"], img_width, img_height):
                if end_index > start_index:
                    second_last_coords = values[end_index - 1]
                    delta_x = last_coords["x"] - second_last_coords["x"]
                    delta_y = last_coords["y"] - second_last_coords["y"]

                    current_x = last_coords["x"]
                    current_y = last_coords["y"]
                    frame_index = end_index + 1

                    while frame_index < len(values):
                        current_x += delta_x
                        current_y += delta_y

                        if is_out_of_bounds(current_x, current_y, img_width, img_height):
                            break

                        cleaned_predictions[frames[frame_index]] = {"x": current_x, "y": current_y}
                        frame_index += 1
        else:
            i += 1

    save_json(cleaned_predictions, output_file)
    print(f"Predizioni estese salvate in: {output_file}")

def process_selected_files(input_files, output_directory, **kwargs):
    """Elabora una lista di file specifici."""
    os.makedirs(output_directory, exist_ok=True)

    for input_file in input_files:
        file_name = os.path.basename(input_file)
        output_file = os.path.join(output_directory, file_name.replace("_Interpolato.json", "_finale.json"))
        extend_predictions(input_file, output_file, **kwargs)

In [ ]:
input_files = [
    "/kaggle/working/model/yolo_output_prog/train/ID-5_Interpolato.json",
    "/kaggle/working/model/yolo_output_prog/train/ID-6_Interpolato.json"
]

output_directory = "/kaggle/working/model/yolo_output_prog/train/"
process_selected_files(input_files, output_directory)

In [ ]:
# Percorsi ai tuoi file
gt_ann_file = '/kaggle/input/newdataset/data/testgt/ID-5.xml'  # Ground truth file
pred_f1ile = '/kaggle/working/model/yolo_output_prog/train/ID-5_finale.json'  # Predizioni in formato JSON

evaluator = TrackingEvaluator(gt_ann_file, pred_file)
evaluator.load_data()
evaluator.compute_frame_indices()
evaluator.evaluate_metrics()
mse1x= evaluator.compute_mse()

In [ ]:
gt_ann_file = '/kaggle/input/newdataset/data/testgt/ID-6.xml'  # Ground truth file
pred_file = '/kaggle/working/model/yolo_output_prog/train/ID-6_finale.json'  # Predizioni in formato JSON
evaluator = TrackingEvaluator(gt_ann_file, pred_file)
evaluator.load_data()
evaluator.compute_frame_indices()
evaluator.evaluate_metrics()
mse2x = evaluator.compute_mse()

In [ ]:
media=np.mean([mse1x, mse2x])
print("Risultato ottenuto:",media)

# Valutazione del Mean Squared Error

|          | *MSE*  |
|:--------:|:------:|
| ID-5     | 0.0041 |
| ID-6     | 0.0162 |
| Mse Medio| 0.0102 |

In [ ]:
# Creazione di una cartella unica per tutti i frame e le etichette
frames_output_dir = "/kaggle/working/model/yolo_output_prog/train/test/ID-5"
labels_output_dir = "/kaggle/working/model/yolo_output_prog/train/no_service"
videos = [
    ("ID-5", "/kaggle/input/newdataset/data/testgt/ID-5.avi", "/kaggle/input/newdataset/data/testgt/ID-5.xml")

]

for video_name, video_path, xml_file in videos:
    convert_xml_and_extract_frames(
        video_path=video_path,
        xml_file=xml_file,
        frames_dir=frames_output_dir,
        labels_dir=labels_output_dir,
        img_width=1920,
        img_height=1088,
        video_name=video_name
    )

print("Tutti i frame e le etichette sono stati salvati.")

In [ ]:
# Creazione di una cartella unica per tutti i frame e le etichette
frames_output_dir = "/kaggle/working/model/yolo_output_prog/train/test/ID-6"
labels_output_dir = "/kaggle/working/model/yolo_output_prog/train/no_service"
videos = [
    ("ID-6", "/kaggle/input/newdataset/data/testgt/ID-6.avi", "/kaggle/input/newdataset/data/testgt/ID-6.xml")

]

for video_name, video_path, xml_file in videos:
    convert_xml_and_extract_frames(
        video_path=video_path,
        xml_file=xml_file,
        frames_dir=frames_output_dir,
        labels_dir=labels_output_dir,
        img_width=1920,
        img_height=1088,
        video_name=video_name
    )

print("Tutti i frame e le etichette sono stati salvati.")

In [ ]:
import cv2
import os
import json
import math

def draw_bounding_boxes_with_trajectory(image_path, prediction, trajectory, max_frames, max_distance=100, color=(0, 0, 255)):
    
    image = cv2.imread(image_path)
    if image is None:
        print(f"Immagine non trovata: {image_path}")
        return None

    height, width, _ = image.shape
    current_position = None

    if prediction["x"] != -1 and prediction["y"] != -1:
        x = prediction["x"]
        y = prediction["y"]
        box_size = 50  # Dimensione del bounding box in pixel
        x1 = int(x - box_size / 2)
        y1 = int(y - box_size / 2)
        x2 = int(x + box_size / 2)
        y2 = int(y + box_size / 2)

        # Salva la posizione corrente per la traiettoria
        current_position = (x, y)

        # Disegna il bounding box
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.putText(image, "Ball", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    # Controllo variazione eccessiva
    if trajectory and current_position:
        last_position = trajectory[-1]
        if last_position is not None:
            distance = math.sqrt((current_position[0] - last_position[0]) ** 2 +
                                 (current_position[1] - last_position[1]) ** 2)
            if distance > max_distance:
                print(f"Variazione eccessiva ignorata: {distance:.2f} pixel")
                current_position = None

    # Disegna la traiettoria
    if trajectory and len(trajectory) > 1:
        for i in range(1, len(trajectory)):
            if trajectory[i - 1] is not None and trajectory[i] is not None:
                pt1 = tuple(map(int, trajectory[i - 1]))
                pt2 = tuple(map(int, trajectory[i]))
                cv2.line(image, pt1, pt2, color, 2)


    # Aggiungi la posizione corrente alla traiettoria
    if current_position:
        trajectory.append(current_position)
    else:
        trajectory.append(None)  # Interrompe la traiettoria se la palla non è visibile

    if len(trajectory) > max_frames:
        trajectory.pop(0)  # Rimuovi i punti più vecchi

    return image

def create_video_with_bboxes_and_trajectory_json(video_config, fps=25, max_trajectory_seconds=0.5):
    """
    Crea video con bounding box e traiettoria sovrapposti usando predizioni da file JSON.
    Ora supporta più video.

    :param video_config: Lista di dizionari con configurazione per ciascun video.
    :param fps: Frame per secondo del video.
    :param max_trajectory_seconds: Numero massimo di secondi da mantenere nella traiettoria.
    """
    for config in video_config:
        image_dir = config["image_dir"]
        json_file = config["json_file"]
        output_video_path = config["output_video_path"]

        # Carica le predizioni
        with open(json_file, "r") as f:
            predictions = json.load(f)

        # Ottieni la lista delle immagini
        image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(".jpg")])

        if not image_files:
            print(f"Nessuna immagine trovata nella directory: {image_dir}")
            continue

        # Leggi la prima immagine per ottenere dimensioni
        first_image_path = os.path.join(image_dir, image_files[0])
        first_image = cv2.imread(first_image_path)
        height, width, _ = first_image.shape

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec per MP4
        video_writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

        trajectory = []  # Lista per salvare la traiettoria
        max_frames = int(max_trajectory_seconds * fps)  # Numero massimo di frame nella traiettoria

        for image_file in image_files:
            image_path = os.path.join(image_dir, image_file)
            frame_number = image_file.split("_frame")[1].split(".")[0]  # Es. "000000"
            json_key = f"{frame_number}.jpg"  # Es. "000000.jpg"

            # Ottieni la predizione dal file JSON
            prediction = predictions.get(json_key, {"x": -1, "y": -1})

            # Disegna bounding box e traiettoria
            annotated_image = draw_bounding_boxes_with_trajectory(image_path, prediction, trajectory, max_frames)
            if annotated_image is not None:
                video_writer.write(annotated_image)

        video_writer.release()
        print(f"Video generato con successo: {output_video_path}")



In [ ]:
# Configurazione per due video
video_config = [
    {
        "image_dir": "/kaggle/working/model/yolo_output_prog/train2/test/ID-5",  # Directory immagini di test
        "json_file": "/kaggle/working/model/yolo_output_prog/train2\ID-5_ultimato.json",  # Predizioni per video 5
        "output_video_path": "/kaggle/working/model/yolo_output_prog/train2\ID-5.mp4"  # Video di output per video 5
    },
    {
        "image_dir": "/kaggle/working/model/yolo_output_prog/train2/test/ID-6",  # Directory immagini di test
        "json_file": '/kaggle/working/model/yolo_output_prog/train2\ID-6_ultimato.json',  # Predizioni per video 6
        "output_video_path": "/kaggle/working/model/yolo_output_prog/train2\ID-6.mp4"  # Video di output per video 6
    }
]

# Crea i video
create_video_with_bboxes_and_trajectory_json(video_config, fps=25, max_trajectory_seconds=0.5)
